> # ⚠️ ARCHIVED — DO NOT CITE ANY NUMBER FROM THIS NOTEBOOK
>
> This notebook is from the project's first generation (2026-08-11). It is kept
> to show the methodological path, **not** as evidence.
>
> - Its stored outputs have been **stripped**, deliberately, so no figure or table
>   here can be mistaken for a current result. Git history retains them.
> - It reads data paths and episode identifiers that **no longer exist**, so it
>   cannot be re-executed to regenerate them.
> - Where it uses the Grand Ouest reference, note that the reference has since been
>   re-resolved onto a new episode reconstruction, and the linkage methods,
>   thresholds, and splits all changed afterwards. Same source data, different
>   everything else.
>
> Current evidence lives in `notebooks/10`–`14`, the `*.md` reports at the
> repository root, and `reports/boamp_methodology_chapter.pdf`.


# BOAMP Linkage Error Analysis

## tl;dr

This notebook diagnoses why the current baseline linkage methods are not yet good enough for survival analysis. It converts predictions into error types, creates review tables for false positives and false negatives, and saves visual evidence for reports.


## Context & Methods

The current best baseline is `M3_weighted_rule_score`, but locked-test precision@1 is only 0.25. This notebook explains that result by inspecting top-1 predictions, score distributions, candidate-pair evidence, and subgroup behavior.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 220)
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/processed/boamp_grand_ouest/linkage_baseline_predictions.parquet").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data/processed/boamp_grand_ouest"
FIGURE_DIR = PROCESSED_DIR / "figures" / "linkage_error_analysis"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = PROCESSED_DIR / "linkage_baseline_predictions.parquet"
PAIRS_PATH = PROCESSED_DIR / "renewal_candidate_pairs.parquet"
ANCHOR_PATH = PROCESSED_DIR / "reference_anchor_episodes.parquet"
LINKS_PATH = PROCESSED_DIR / "reference_successor_links.parquet"
CONFIG_PATH = PROCESSED_DIR / "linkage_best_baseline_config.json"
ERROR_CASES_PATH = PROCESSED_DIR / "linkage_error_cases.csv"
FALSE_POSITIVE_PATH = PROCESSED_DIR / "linkage_false_positive_review_cases.csv"
FALSE_NEGATIVE_PATH = PROCESSED_DIR / "linkage_false_negative_review_cases.csv"
SUMMARY_PATH = PROCESSED_DIR / "linkage_error_analysis_summary.json"

print(PROJECT_ROOT)


## Data

### 1. Load Predictions, Candidate Pairs, And Reference Truth


In [ ]:
def parse_json_list(value) -> list:
    if value is None or pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    try:
        parsed = json.loads(text)
    except Exception:
        return []
    return parsed if isinstance(parsed, list) else [parsed]

predictions = pd.read_parquet(PREDICTIONS_PATH)
pairs = pd.read_parquet(PAIRS_PATH)
anchors = pd.read_parquet(ANCHOR_PATH)
links = pd.read_parquet(LINKS_PATH)
config = json.loads(CONFIG_PATH.read_text())
best_algorithm = config["best_algorithm_by_locked_precision_then_recall5"]
best_config = config["best_algorithm_config"]

best_predictions = predictions[predictions["algorithm"].eq(best_algorithm)].copy()
best_predictions["predicted_idwebs"] = best_predictions["predicted_idwebs_json"].map(parse_json_list)
best_predictions["true_successor_idwebs"] = best_predictions["true_successor_idwebs_json"].map(parse_json_list)
best_predictions["predicted_any"] = best_predictions["predicted_idwebs"].map(bool)
best_predictions["has_true_successor"] = best_predictions["true_successor_idwebs"].map(bool)
best_predictions["top1_predicted_idweb"] = best_predictions["top1_predicted_idweb"].fillna("").astype(str)

print("Best algorithm:", best_algorithm, best_config)
print(best_predictions.shape, pairs.shape, anchors.shape, links.shape)


### 2. Reconstruct Best Baseline Scores For Evidence Inspection


In [ ]:
WEIGHT_PRESETS = {
    "balanced": {"buyer": 0.35, "text": 0.25, "cpv": 0.20, "time": 0.15, "geo": 0.05},
    "precision_buyer": {"buyer": 0.45, "text": 0.20, "cpv": 0.20, "time": 0.10, "geo": 0.05},
    "text_heavy": {"buyer": 0.30, "text": 0.35, "cpv": 0.15, "time": 0.15, "geo": 0.05},
}

def add_weighted_score(frame: pd.DataFrame, weights: dict, score_name: str) -> pd.DataFrame:
    tmp = frame.copy()
    buyer_score = tmp["buyer_match_type"].map({"siren": 1.0, "normalized_name": 0.9, "fuzzy_name": 0.7, "token_overlap": 0.6}).fillna(0)
    text_score = tmp[["word_tfidf_similarity", "char_ngram_tfidf_similarity"]].max(axis=1).fillna(0).clip(0, 1)
    cpv_score = tmp["cpv_overlap_score"].fillna(0).clip(0, 1)
    time_score = tmp["duration_gap_score"].fillna(0).clip(0, 1)
    geo_score = tmp["same_region"].astype(bool).astype(float)
    tmp["buyer_score_component"] = buyer_score
    tmp["text_score_component"] = text_score
    tmp["cpv_score_component"] = cpv_score
    tmp["time_score_component"] = time_score
    tmp["geo_score_component"] = geo_score
    tmp[score_name] = 100 * (weights["buyer"] * buyer_score + weights["text"] * text_score + weights["cpv"] * cpv_score + weights["time"] * time_score + weights["geo"] * geo_score)
    return tmp

scored_pairs = add_weighted_score(pairs, WEIGHT_PRESETS[best_config["weights"]], "best_weighted_score")
top1_evidence = (
    best_predictions[["sample_id", "benchmark_split", "anchor_episode_id", "final_outcome", "true_successor_idwebs", "predicted_idwebs", "predicted_any", "has_true_successor", "top1_predicted_idweb", "top1_correct", "top5_correct"]]
    .merge(scored_pairs, left_on=["sample_id", "top1_predicted_idweb"], right_on=["sample_id", "candidate_idweb"], how="left", suffixes=("", "_pair"))
)

def classify_error(row) -> str:
    if row["has_true_successor"] and row["top1_correct"]:
        return "true_positive_top1"
    if row["has_true_successor"] and row["top5_correct"]:
        return "true_successor_in_top5_not_top1"
    if row["has_true_successor"] and row["predicted_any"]:
        return "false_negative_wrong_link"
    if row["has_true_successor"] and not row["predicted_any"]:
        return "false_negative_no_link"
    if not row["has_true_successor"] and row["predicted_any"]:
        return "false_positive_no_successor"
    return "true_negative_no_link"

top1_evidence["error_type"] = top1_evidence.apply(classify_error, axis=1)
print(top1_evidence["error_type"].value_counts().to_string())


## Results

### 3. Error Type Summary


In [ ]:
error_summary = (
    top1_evidence.groupby(["benchmark_split", "error_type"])
    .size()
    .reset_index(name="anchors")
    .sort_values(["benchmark_split", "anchors"], ascending=[True, False])
)
display(error_summary)

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=error_summary, x="error_type", y="anchors", hue="benchmark_split", ax=ax)
ax.set_title("Best baseline error types by benchmark split")
ax.set_xlabel("")
ax.set_ylabel("Anchor episodes")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
path_error_types = FIGURE_DIR / "01_error_types_by_split.png"
fig.savefig(path_error_types, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()


### 4. Evidence Feature Distributions By Error Type


In [ ]:
feature_cols = ["best_weighted_score", "buyer_score_component", "text_score_component", "cpv_score_component", "time_score_component", "buyer_name_similarity", "word_tfidf_similarity", "char_ngram_tfidf_similarity", "time_gap_months"]
plot_frame = top1_evidence[top1_evidence["benchmark_split"].eq("LOCKED_TEST")].copy()

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
for ax, col in zip(axes.ravel(), ["best_weighted_score", "text_score_component", "cpv_score_component", "time_gap_months"]):
    sns.boxplot(data=plot_frame, x="error_type", y=col, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=35)
    ax.set_xlabel("")
fig.tight_layout()
path_feature_boxes = FIGURE_DIR / "02_feature_distributions_by_error_type.png"
fig.savefig(path_feature_boxes, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

summary_features = plot_frame.groupby("error_type")[feature_cols].median(numeric_only=True).round(3)
display(summary_features)


### 5. False Positive Review Cases


In [ ]:
review_columns = [
    "sample_id", "benchmark_split", "anchor_episode_id", "final_outcome", "final_confidence", "anchor_award_notice_date", "anchor_expected_end_date",
    "anchor_buyer_name", "anchor_theme", "anchor_text_normalized", "top1_predicted_idweb", "candidate_dateparution", "candidate_buyer_name",
    "candidate_primary_cpv_prefix2", "candidate_technology_segment", "candidate_text_normalized", "url_avis", "time_gap_months", "buyer_match_type",
    "buyer_name_similarity", "buyer_token_overlap", "same_cpv_prefix", "cpv_overlap_score", "same_theme", "word_tfidf_similarity",
    "char_ngram_tfidf_similarity", "duration_gap_score", "best_weighted_score", "error_type"
]
false_positive_cases = top1_evidence[top1_evidence["error_type"].eq("false_positive_no_successor")].copy()
false_positive_cases = false_positive_cases[[c for c in review_columns if c in false_positive_cases.columns]].sort_values("best_weighted_score", ascending=False)
false_positive_cases.to_csv(FALSE_POSITIVE_PATH, index=False, encoding="utf-8")
display(false_positive_cases.head(15))
print(FALSE_POSITIVE_PATH)


### 6. False Negative And Missed-Ranking Review Cases


In [ ]:
false_negative_cases = top1_evidence[top1_evidence["error_type"].isin(["false_negative_wrong_link", "false_negative_no_link", "true_successor_in_top5_not_top1"])].copy()
false_negative_cases["true_successor_idwebs_json"] = false_negative_cases["true_successor_idwebs"].map(lambda values: json.dumps(values, ensure_ascii=False))
false_negative_cases["predicted_idwebs_json"] = false_negative_cases["predicted_idwebs"].map(lambda values: json.dumps(values, ensure_ascii=False))
false_negative_cases = false_negative_cases[[c for c in ["sample_id", "benchmark_split", "anchor_episode_id", "final_outcome", "anchor_award_notice_date", "anchor_buyer_name", "anchor_theme", "anchor_text_normalized", "true_successor_idwebs_json", "predicted_idwebs_json", "top1_predicted_idweb", "candidate_dateparution", "candidate_buyer_name", "candidate_text_normalized", "best_weighted_score", "word_tfidf_similarity", "char_ngram_tfidf_similarity", "cpv_overlap_score", "duration_gap_score", "error_type"] if c in false_negative_cases.columns]].sort_values(["benchmark_split", "error_type", "sample_id"])
false_negative_cases.to_csv(FALSE_NEGATIVE_PATH, index=False, encoding="utf-8")
display(false_negative_cases.head(20))
print(FALSE_NEGATIVE_PATH)


### 7. Subgroup Error Diagnostics


In [ ]:
subgroup = top1_evidence.copy()
subgroup["anchor_has_siren"] = subgroup["anchor_buyer_siren"].fillna("").astype(str).str.len().gt(0)
subgroup["anchor_has_expected_end"] = subgroup["anchor_expected_end_date"].fillna("").astype(str).str.len().gt(0)
subgroup["anchor_theme"] = subgroup["anchor_theme"].fillna("unknown")
subgroup_metrics = []
for group_col in ["anchor_has_siren", "anchor_has_expected_end", "anchor_theme", "benchmark_split"]:
    for group_value, group in subgroup.groupby(group_col, dropna=False):
        predicted = group["predicted_any"].sum()
        positives = group["has_true_successor"].sum()
        no_successors = (~group["has_true_successor"]).sum()
        subgroup_metrics.append({
            "group": group_col,
            "value": str(group_value),
            "anchors": int(len(group)),
            "precision_at_1": float(group["top1_correct"].sum() / predicted) if predicted else 0.0,
            "recall_at_5": float(group["top5_correct"].sum() / positives) if positives else np.nan,
            "false_positive_rate_no_successor": float(((~group["has_true_successor"]) & group["predicted_any"]).sum() / no_successors) if no_successors else np.nan,
            "coverage_rate": float(group["predicted_any"].mean()) if len(group) else 0.0,
        })
subgroup_df = pd.DataFrame(subgroup_metrics)
display(subgroup_df)

fig, ax = plt.subplots(figsize=(9.5, 5.2))
theme_plot = subgroup_df[subgroup_df["group"].eq("anchor_theme")].sort_values("precision_at_1", ascending=False)
sns.barplot(data=theme_plot, x="value", y="precision_at_1", color="#4E79A7", ax=ax)
ax.set_title("Precision@1 by anchor CPV theme")
ax.set_xlabel("Anchor theme")
ax.set_ylabel("Precision@1")
ax.set_ylim(0, 1)
fig.tight_layout()
path_theme_precision = FIGURE_DIR / "03_precision_by_anchor_theme.png"
fig.savefig(path_theme_precision, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()


### 8. Save Error Analysis Summary


In [ ]:
top1_evidence[[c for c in review_columns if c in top1_evidence.columns]].to_csv(ERROR_CASES_PATH, index=False, encoding="utf-8")
summary = {
    "created_at": pd.Timestamp.now().isoformat(timespec="seconds"),
    "best_algorithm": best_algorithm,
    "best_config": best_config,
    "inputs": {"predictions": str(PREDICTIONS_PATH), "candidate_pairs": str(PAIRS_PATH)},
    "outputs": {
        "error_cases": str(ERROR_CASES_PATH),
        "false_positive_cases": str(FALSE_POSITIVE_PATH),
        "false_negative_cases": str(FALSE_NEGATIVE_PATH),
        "figure_dir": str(FIGURE_DIR),
    },
    "error_type_counts": {str(k): int(v) for k, v in top1_evidence["error_type"].value_counts().items()},
    "locked_test_error_type_counts": {str(k): int(v) for k, v in top1_evidence.loc[top1_evidence["benchmark_split"].eq("LOCKED_TEST"), "error_type"].value_counts().items()},
    "locked_test_feature_medians_by_error_type": summary_features.reset_index().to_dict(orient="records"),
    "main_diagnosis": "The current weighted baseline retrieves many true successors but over-links no-successor anchors. It should be made stricter before survival analysis.",
    "recommended_next_changes": [
        "Increase acceptance threshold or add a high-confidence threshold tier.",
        "Require stronger text similarity for no-duration or normalized-name-only matches.",
        "Penalize token-overlap/fuzzy buyer matches unless CPV and text evidence are both strong.",
        "Separate automatic links from manual-review candidates.",
    ],
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

manifest = pd.DataFrame([
    ["error_types_by_split", str(path_error_types)],
    ["feature_distributions_by_error_type", str(path_feature_boxes)],
    ["precision_by_anchor_theme", str(path_theme_precision)],
], columns=["artifact", "path"])
display(manifest)
assert ERROR_CASES_PATH.exists()
assert FALSE_POSITIVE_PATH.exists()
assert FALSE_NEGATIVE_PATH.exists()
assert SUMMARY_PATH.exists()


## Takeaways

The current baseline is a useful scaffold, but it should not feed survival analysis yet. The next implementation should use this error evidence to create stricter high-confidence acceptance rules and separate automatic links from manual-review candidates.
